In [2]:
import pandas as pd

df = pd.read_csv(
    "train_v2.csv",
    dtype={"fullVisitorId": str},
    nrows=5
)

print(df)

  channelGrouping                              customDimensions      date  \
0  Organic Search             [{'index': '4', 'value': 'EMEA'}]  20171016   
1        Referral    [{'index': '4', 'value': 'North America'}]  20171016   
2          Direct    [{'index': '4', 'value': 'North America'}]  20171016   
3  Organic Search             [{'index': '4', 'value': 'EMEA'}]  20171016   
4  Organic Search  [{'index': '4', 'value': 'Central America'}]  20171016   

                                              device        fullVisitorId  \
0  {"browser": "Firefox", "browserVersion": "not ...  3162355547410993243   
1  {"browser": "Chrome", "browserVersion": "not a...  8934116514970143966   
2  {"browser": "Chrome", "browserVersion": "not a...  7992466427990357681   
3  {"browser": "Chrome", "browserVersion": "not a...  9075655783635761930   
4  {"browser": "Chrome", "browserVersion": "not a...  6960673291025684308   

                                          geoNetwork  \
0  {"continent": "

# Import thư viện và cấu hình 
Block setup ban đầu để chuẩn bị xử lý file Google Analytics rất lớn mà không làm đầy RAM

In [2]:
# ============================================================
# Google Analytics Revenue Prediction - Full Processing Pipeline
# ============================================================

import os
import gc
import json
import ast
import numpy as np
import pandas as pd
from pandas import json_normalize
from tqdm import tqdm

# =========================
# Block 1 — Config
# =========================

INPUT_FILE = "train_v2.csv"
OUTPUT_DIR = "processed_chunks"
OUTPUT_PREFIX = "train"

CHUNK_SIZE = 100_000

ID_COL = "fullVisitorId"

JSON_COLUMNS = ["device", "geoNetwork", "totals", "trafficSource"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Đọc dữ liệu an toàn
Mục đích chính: chuyển dữ liệu đang ở dạng chuỗi thành Python object thật, nhưng nếu lỗi thì không làm chương trình bị crash.

In [10]:
# =========================
# Block 2 — Safe parsers
# =========================

def safe_json_loads(x):
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    try:
        return json.loads(x)
    except Exception:
        try:
            return ast.literal_eval(x)
        except Exception:
            return {}


def safe_list_loads(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        try:
            return json.loads(x)
        except Exception:
            return []

# Tách JSON thành nhiều cột thường
Block này dùng để tách các cột JSON lồng nhau thành nhiều cột thường trong DataFrame.

In [11]:
# =========================
# Block 3 — Flatten JSON columns
# =========================

def flatten_json_columns(df):
    for col in JSON_COLUMNS:
        if col in df.columns:
            parsed = df[col].apply(safe_json_loads)
            col_df = json_normalize(parsed)

            col_df.columns = [f"{col}_{subcol}" for subcol in col_df.columns]

            df = df.drop(columns=[col]).join(col_df)

    return df

# Xử lý cột customDimensions trong dataset Google Analytics.
Khác với các cột như device, geoNetwork, totals, trafficSource, cột customDimensions không phải là JSON object đơn, mà là list chứa dictionary.

In [12]:
# =========================
# Block 4 — Process customDimensions
# =========================

def process_custom_dimensions(df):
    if "customDimensions" not in df.columns:
        return df

    custom = df["customDimensions"].apply(safe_list_loads)

    df["customDimensions_count"] = custom.apply(len)

    df["customDimensions_index"] = custom.apply(
        lambda x: x[0].get("index") if len(x) > 0 and isinstance(x[0], dict) else np.nan
    )

    df["customDimensions_value"] = custom.apply(
        lambda x: x[0].get("value") if len(x) > 0 and isinstance(x[0], dict) else np.nan
    )

    df = df.drop(columns=["customDimensions"])

    return df

# Gợi ý nhưng không sử dụng: Xử lý cột hits
Block này dùng để xử lý cột hits trong Google Analytics.

hits là một cột rất nặng vì nó chứa toàn bộ chuỗi hành vi trong một session, ví dụ người dùng xem trang nào, click gì, có event không, có transaction không, thời gian từng hit là bao nhiêu.

Thay vì bung toàn bộ hits ra thành hàng trăm hoặc hàng nghìn cột, code này chỉ lấy các feature tổng hợp quan trọng.

In [13]:
# =========================
# Block 5 — Process hits
# =========================
# hits rất nặng, không bung toàn bộ.
# Ta trích feature tổng hợp để vẫn giữ thông tin hành vi.

def process_hits(df):
    if "hits" not in df.columns:
        return df

    hits = df["hits"].apply(safe_list_loads)

    df["hits_count"] = hits.apply(len)

    df["hits_first_type"] = hits.apply(
        lambda x: x[0].get("type") if len(x) > 0 and isinstance(x[0], dict) else np.nan
    )

    df["hits_has_event"] = hits.apply(
        lambda x: int(any(isinstance(h, dict) and h.get("type") == "EVENT" for h in x))
    )

    df["hits_has_page"] = hits.apply(
        lambda x: int(any(isinstance(h, dict) and h.get("type") == "PAGE" for h in x))
    )

    df["hits_has_transaction"] = hits.apply(
        lambda x: int(any(
            isinstance(h, dict) and h.get("transaction") not in [None, {}, ""]
            for h in x
        ))
    )

    df["hits_total_time"] = hits.apply(
        lambda x: sum(
            int(h.get("time", 0))
            for h in x
            if isinstance(h, dict) and str(h.get("time", "0")).isdigit()
        )
    )

    df["hits_max_time"] = hits.apply(
        lambda x: max(
            [
                int(h.get("time", 0))
                for h in x
                if isinstance(h, dict) and str(h.get("time", "0")).isdigit()
            ],
            default=0
        )
    )

    df = df.drop(columns=["hits"])

    return df

# Chuẩn hóa các giá trị missing kì lạ
Trong dataset Google Analytics, dữ liệu thiếu không phải lúc nào cũng hiện là NaN. Nó có thể được ghi bằng nhiều chuỗi khác nhau như:

(not set)
(not provided)
unknown.unknown
not available in demo dataset
(none)
""

Nếu không xử lý, model hoặc bước phân tích sẽ hiểu nhầm các chuỗi này là giá trị thật.

In [14]:
# =========================
# Block 6 — Clean weird NA values
# =========================

def clean_weird_na(df):
    weird_values = [
        "unknown.unknown",
        "(not set)",
        "not available in demo dataset",
        "(not provided)",
        "(none)",
        "<NA>",
        ""
    ]

    df = df.replace(weird_values, pd.NA)

    return df

# Điền missing values cơ bản
Block này dùng để điền missing value theo ý nghĩa của từng loại cột, thay vì điền tất cả missing bằng một giá trị giống nhau.

Sau Block 6, các giá trị thiếu đã được chuẩn hóa thành pd.NA. Sang Block 7, code quyết định:

Cột nào thiếu thì điền "NA",
Cột nào thiếu thì điền 0,
Cột nào thiếu thì điền True,
Cột nào thiếu thì điền False

In [15]:
# =========================
# Block 7 — Fill NA by semantic type
# =========================

def fill_na_by_semantic_type(df):
    to_na_cols = [
        "trafficSource_adContent",
        "trafficSource_adwordsClickInfo.adNetworkType",
        "trafficSource_adwordsClickInfo.slot",
        "trafficSource_adwordsClickInfo.gclId",
        "trafficSource_keyword",
        "trafficSource_referralPath",
        "customDimensions_value",
    ]

    to_0_cols = [
        "totals_transactionRevenue",
        "totals_totalTransactionRevenue",
        "totals_sessionQualityDim",
        "totals_bounces",
        "totals_timeOnSite",
        "totals_newVisits",
        "totals_pageviews",
        "totals_transactions",
        "customDimensions_index",
        "hits_count",
        "hits_has_event",
        "hits_has_page",
        "hits_has_transaction",
        "hits_total_time",
        "hits_max_time",
    ]

    to_true_cols = [
        "trafficSource_adwordsClickInfo.isVideoAd"
    ]

    to_false_cols = [
        "trafficSource_isTrueDirect"
    ]

    for col in to_na_cols:
        if col in df.columns:
            df[col] = df[col].fillna("NA")

    for col in to_0_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    for col in to_true_cols:
        if col in df.columns:
            df[col] = df[col].fillna(True)

    for col in to_false_cols:
        if col in df.columns:
            df[col] = df[col].fillna(False)

    return df

# Tạo thêm các feature thời gian
Block này dùng để tạo thêm các feature thời gian từ 2 cột gốc:

date,
visitStartTime

Mục đích: biến thông tin ngày/giờ thành các cột dễ dùng cho phân tích và machine learning.

In [16]:
# =========================
# Block 8 — Date/time features
# =========================

def encode_date_features(df):
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d", errors="coerce")

        df["Date_Year"] = df["date"].dt.year
        df["Date_Month"] = df["date"].dt.month
        df["Date_Day"] = df["date"].dt.day
        df["Date_Dayofweek"] = df["date"].dt.dayofweek
        df["Date_Dayofyear"] = df["date"].dt.dayofyear
        df["Date_Week"] = df["date"].dt.isocalendar().week.astype("Int64")
        df["Date_Is_month_end"] = df["date"].dt.is_month_end.astype(int)
        df["Date_Is_month_start"] = df["date"].dt.is_month_start.astype(int)
        df["Date_Is_quarter_end"] = df["date"].dt.is_quarter_end.astype(int)
        df["Date_Is_quarter_start"] = df["date"].dt.is_quarter_start.astype(int)
        df["Date_Is_year_end"] = df["date"].dt.is_year_end.astype(int)
        df["Date_Is_year_start"] = df["date"].dt.is_year_start.astype(int)

    if "visitStartTime" in df.columns:
        df["visitStartTime_datetime"] = pd.to_datetime(
            df["visitStartTime"],
            unit="s",
            errors="coerce"
        )

        df["Date_Hour"] = df["visitStartTime_datetime"].dt.hour

    return df

# Sửa kiểu dữ liệu số
Block 9 dùng để sửa kiểu dữ liệu của các cột số.

Sau khi đọc từ CSV và flatten JSON, nhiều cột nhìn như số nhưng thực chất đang ở dạng object hoặc string.

- Ví dụ:

totals_hits = "5"

totals_pageviews = "3"

totals_transactionRevenue = "123000000"

Về mặt hình thức, chúng là chuỗi. Nếu không đổi sang số thật, sẽ khó tính toán, thống kê, train model hoặc vẽ biểu đồ.

In [17]:
# =========================
# Block 9 — Fix numeric types
# =========================

def fix_numeric_types(df):
    numeric_cols = [
        "visitId",
        "visitNumber",
        "visitStartTime",

        "totals_visits",
        "totals_hits",
        "totals_pageviews",
        "totals_bounces",
        "totals_newVisits",
        "totals_transactionRevenue",
        "totals_totalTransactionRevenue",
        "totals_sessionQualityDim",
        "totals_timeOnSite",
        "totals_transactions",

        "trafficSource_adwordsClickInfo.page",

        "customDimensions_index",
        "customDimensions_count",

        "hits_count",
        "hits_has_event",
        "hits_has_page",
        "hits_has_transaction",
        "hits_total_time",
        "hits_max_time",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    int_cols = [
        "totals_visits",
        "totals_hits",
        "totals_pageviews",
        "totals_bounces",
        "totals_newVisits",
        "totals_sessionQualityDim",
        "totals_timeOnSite",
        "totals_transactions",
        "trafficSource_adwordsClickInfo.page",
        "customDimensions_index",
        "customDimensions_count",
        "hits_count",
        "hits_has_event",
        "hits_has_page",
        "hits_has_transaction",
        "hits_total_time",
        "hits_max_time",
    ]

    for col in int_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)

    if "totals_transactionRevenue" in df.columns:
        df["totals_transactionRevenue"] = df["totals_transactionRevenue"].fillna(0).astype(float)

    if "totals_totalTransactionRevenue" in df.columns:
        df["totals_totalTransactionRevenue"] = df["totals_totalTransactionRevenue"].fillna(0).astype(float)

    return df

# Tạo biến liên quan đến Revenue
Block 10 làm 3 việc:

1. Tạo cột doanh thu sạch: transactionRevenue
2. Tạo cột doanh thu dễ đọc hơn: transactionRevenue_dollar (chia cho 1 triệu)
3. Tạo target log revenue để train model: target_log_revenue

Nói ngắn gọn: block này biến revenue gốc thành các biến doanh thu dùng được cho visualization và machine learning.

In [18]:
# =========================
# Block 10 — Revenue features
# =========================

def create_revenue_features(df):
    if "totals_transactionRevenue" in df.columns:
        df["transactionRevenue"] = df["totals_transactionRevenue"].fillna(0)

        # Dùng để visualize dễ hiểu
        df["transactionRevenue_dollar"] = df["transactionRevenue"] / 1_000_000

        # Target phổ biến cho bài này
        df["target_log_revenue"] = np.log1p(df["transactionRevenue"])

    return df

# Xóa cột vô nghĩa
Xóa các cột đã biết trước là ít hữu ích, hầu như constant, chứa thông tin demo không dùng được, hoặc quá chi tiết/nặng.

- Nói ngắn gọn:

Giữ lại những cột có khả năng dự đoán revenue

Bỏ những cột rác/constant/heavy để DataFrame gọn hơn

In [19]:
# =========================
# Block 11 — Drop known useless/heavy columns
# =========================

def drop_known_useless_columns(df):
    known_const_cols = [
        "socialEngagementType",
        "device_browserSize",
        "device_browserVersion",
        "device_flashVersion",
        "device_language",
        "device_mobileDeviceBranding",
        "device_mobileDeviceInfo",
        "device_mobileDeviceMarketingName",
        "device_mobileDeviceModel",
        "device_mobileInputSelector",
        "device_operatingSystemVersion",
        "device_screenColors",
        "device_screenResolution",
        "geoNetwork_cityId",
        "geoNetwork_latitude",
        "geoNetwork_longitude",
        "geoNetwork_networkLocation",
        "trafficSource_adwordsClickInfo.criteriaParameters",
        "trafficSource_campaignCode",
    ]

    drop_cols = [col for col in known_const_cols if col in df.columns]

    if drop_cols:
        df = df.drop(columns=drop_cols)

    return df

# Xóa các cột constant (có duy nhất 1 giá trị)

In [20]:
# =========================
# Block 12 — Drop constant columns inside one chunk
# =========================
# Lưu ý: chỉ nên dùng nhẹ.
# Nếu muốn drop chính xác toàn dataset, nên làm pass thứ 2 sau khi xử lý xong.

def drop_constant_columns_in_chunk(df, keep_cols=None):
    if keep_cols is None:
        keep_cols = [
            "totals_bounces",
            "totals_newVisits",
            "target_log_revenue",
            "transactionRevenue",
        ]

    const_cols = []

    for col in df.columns:
        if col in keep_cols:
            continue

        if df[col].nunique(dropna=True) <= 1 and df[col].isna().sum() == 0:
            const_cols.append(col)

    if const_cols:
        df = df.drop(columns=const_cols)

    return df, const_cols

# Gom toàn bộ các bước xử lý trước đó vào một hàm duy nhất để xử lý một chunk dữ liệu.

In [25]:
# =========================
# Block 13 — Process one chunk
# =========================

def process_one_chunk(df, drop_const_in_chunk=False):
    df = df.copy()
    df.reset_index(drop=True, inplace=True)

    df = flatten_json_columns(df)
    df = process_custom_dimensions(df)
    if "hits" in df.columns:
        df = df.drop(columns=["hits"])

    df = clean_weird_na(df)
    df = fill_na_by_semantic_type(df)

    df = encode_date_features(df)
    df = fix_numeric_types(df)
    df = create_revenue_features(df)

    df = drop_known_useless_columns(df)

    const_cols = []

    if drop_const_in_chunk:
        df, const_cols = drop_constant_columns_in_chunk(df)

    return df, const_cols

# Hàm chạy toàn bộ pipeline trên file CSV lớn, đọc file theo từng chunk, xử lý từng chunk, rồi lưu từng chunk đã xử lý ra file .pkl.

In [26]:
# =========================
# Block 14 — Main chunk pipeline
# =========================

def process_csv_to_pickles(
    input_file=INPUT_FILE,
    output_dir=OUTPUT_DIR,
    output_prefix=OUTPUT_PREFIX,
    chunk_size=CHUNK_SIZE,
    drop_const_in_chunk=False
):
    os.makedirs(output_dir, exist_ok=True)

    reader = pd.read_csv(
        input_file,
        dtype={ID_COL: str},
        chunksize=chunk_size,
        low_memory=False
    )

    all_const_cols = []

    for idx, chunk in enumerate(reader):
        print(f"\nProcessing chunk {idx} | raw shape = {chunk.shape}")

        processed_chunk, const_cols = process_one_chunk(
            chunk,
            drop_const_in_chunk=drop_const_in_chunk
        )

        output_path = os.path.join(output_dir, f"{output_prefix}_{idx}.pkl")
        processed_chunk.to_pickle(output_path)

        print(f"Saved: {output_path}")
        print(f"Processed shape = {processed_chunk.shape}")

        if const_cols:
            all_const_cols.extend(const_cols)

        del chunk
        del processed_chunk
        gc.collect()

    print("\nDone!")
    return all_const_cols

In [27]:
# =========================
# Block 15 — Run for train_v2.csv
# =========================

const_cols_seen = process_csv_to_pickles(
    input_file="train_v2.csv",
    output_dir="processed_chunks",
    output_prefix="train",
    chunk_size=100_000,
    drop_const_in_chunk=False
)


Processing chunk 0 | raw shape = (100000, 13)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_56692\126532537.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(True)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_56692\126532537.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)


Saved: processed_chunks\train_0.pkl
Processed shape = (100000, 59)

Processing chunk 1 | raw shape = (100000, 13)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_56692\126532537.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(True)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_56692\126532537.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)


Saved: processed_chunks\train_1.pkl
Processed shape = (100000, 59)

Processing chunk 2 | raw shape = (100000, 13)


...

Saved: processed_chunks\train_15.pkl
Processed shape = (100000, 59)

Processing chunk 16 | raw shape = (100000, 13)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_56692\126532537.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(True)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_56692\126532537.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)


Saved: processed_chunks\train_16.pkl
Processed shape = (100000, 59)

Processing chunk 17 | raw shape = (8337, 13)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_56692\126532537.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(True)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_56692\126532537.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)


Saved: processed_chunks\train_17.pkl
Processed shape = (8337, 59)

Done!


In [4]:
# =========================
# Block 17 — Load processed pickle chunks
# =========================
OUTPUT_DIR = r"D:\data_driven_marketing\processed_chunks"
def load_processed_data(prefix="train", n=None, output_dir=OUTPUT_DIR):
    files = sorted([
        f for f in os.listdir(output_dir)
        if f.startswith(prefix + "_") and f.endswith(".pkl")
    ])

    if n is not None:
        files = files[:n]

    parts = []

    for file in files:
        path = os.path.join(output_dir, file)
        print("Loading:", path)

        part = pd.read_pickle(path)
        parts.append(part)

    df = pd.concat(parts, ignore_index=True)

    return df


# Test load 1 chunk
df_check = load_processed_data(prefix="train", n=18)
print(df_check.shape)
print(df_check.head())

Loading: D:\data_driven_marketing\processed_chunks\train_0.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_1.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_10.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_11.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_12.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_13.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_14.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_15.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_16.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_17.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_2.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_3.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_4.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_5.pkl
Loading: D:\data_driven_marketing\processed_chunks\train_6.pkl
Loading: D:\data_driven_marketing\processed_chu

In [30]:
# =========================
# Block 19 — Check remaining JSON-like columns
# =========================

def check_json_like_columns(df):
    object_cols = df.select_dtypes(include=["object"]).columns.tolist()

    possible_json_cols = []

    for col in object_cols:
        sample = df[col].dropna().astype(str).head(50)

        if sample.str.startswith("{").any() or sample.str.startswith("[").any():
            possible_json_cols.append(col)

    return possible_json_cols


possible_json_cols = check_json_like_columns(df_check)

print("Possible remaining JSON-like columns:")
print(possible_json_cols)

Possible remaining JSON-like columns:
[]


In [31]:
# =========================
# Block 20 — Inspect final columns
# =========================

print("Number of columns:", len(df_check.columns))

for col in df_check.columns:
    print(col)

Number of columns: 59
channelGrouping
date
fullVisitorId
visitId
visitNumber
visitStartTime
device_browser
device_operatingSystem
device_isMobile
device_deviceCategory
geoNetwork_continent
geoNetwork_subContinent
geoNetwork_country
geoNetwork_region
geoNetwork_metro
geoNetwork_city
geoNetwork_networkDomain
totals_visits
totals_hits
totals_pageviews
totals_bounces
totals_newVisits
totals_sessionQualityDim
totals_timeOnSite
totals_transactions
totals_transactionRevenue
totals_totalTransactionRevenue
trafficSource_campaign
trafficSource_source
trafficSource_medium
trafficSource_keyword
trafficSource_referralPath
trafficSource_isTrueDirect
trafficSource_adContent
trafficSource_adwordsClickInfo.page
trafficSource_adwordsClickInfo.slot
trafficSource_adwordsClickInfo.gclId
trafficSource_adwordsClickInfo.adNetworkType
trafficSource_adwordsClickInfo.isVideoAd
customDimensions_count
customDimensions_index
customDimensions_value
Date_Year
Date_Month
Date_Day
Date_Dayofweek
Date_Dayofyear
Date_Wee

In [5]:
df_check.to_pickle("df_full.pkl")

print("Saved to df_full.pkl")

Saved to df_full.pkl
